# Notebook 04b — Comparação complementar com as categorias separadas

O Notebook 04 compara os três métodos tratando sexo, raça/cor e pretensão futura como atributos únicos. Este notebook repete a comparação com cada categoria como uma coluna separada em relação à categoria de referência. O objetivo é verificar se as conclusões dependem da forma de codificar esses atributos.

**Unidades comparadas:** 25 — as 17 variáveis numéricas e ordinais e 8 categorias (sexo feminino; raça/cor parda, preta, amarela e indígena; pretensão "estudar e trabalhar", "ainda não sabe" e "somente trabalhar"). As categorias de referência são sexo masculino, raça/cor branca e pretensão "somente estudar".

**Procedimentos:** os mesmos do Notebook 04, com a mesma semente, a mesma partição por escola e os mesmos hiperparâmetros da floresta escolhidos no notebook 4 (profundidade máxima 12 e mínimo de 5 estudantes por folha).

**Entrada:** `base_analitica.parquet`. **Saída:** `tabelas_comparacao_categorias.xlsx`.

## 0. Configuração

In [1]:
import sys, time, numpy as np, pandas as pd
import sklearn, statsmodels, statsmodels.formula.api as smf
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import r2_score
from scipy.stats import spearmanr
import warnings; warnings.filterwarnings('ignore')

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PASTA = '/content/drive/MyDrive/dissertacao/'   # ajuste para a sua pasta
except ImportError:
    PASTA = './'

SEMENTE = 2025
PARAMS_RF = dict(n_estimators=300, max_depth=12, min_samples_leaf=5, max_features=1/3, n_jobs=-1)  # escolhidos no Notebook 04
TOP = 5
print({'python': sys.version.split()[0], 'scikit-learn': sklearn.__version__, 'statsmodels': statsmodels.__version__})

base = pd.read_parquet(PASTA + 'base_analitica.parquet').reset_index(drop=True)
assert len(base) == 22996
y = base['proficiencia'].values

Mounted at /content/drive
{'python': '3.13.16', 'scikit-learn': '1.6.1', 'statsmodels': '0.15.0'}


## 1. Unidades de comparação

In [2]:
CATEGORIAS = {'fem': ('sexo', 'Feminino', 'Sexo: feminino'),
              'parda': ('raca_cor', 'Parda', 'Raça/cor: parda'),
              'preta': ('raca_cor', 'Preta', 'Raça/cor: preta'),
              'amarela': ('raca_cor', 'Amarela', 'Raça/cor: amarela'),
              'indigena': ('raca_cor', 'Indígena', 'Raça/cor: indígena'),
              'est_trab': ('pretensao_futura', 'Estudar e trabalhar', 'Pretensão: estudar e trabalhar'),
              'nao_sabe_pret': ('pretensao_futura', 'Ainda não sabe', 'Pretensão: ainda não sabe'),
              'so_trab': ('pretensao_futura', 'Somente trabalhar', 'Pretensão: somente trabalhar')}
for c, (v, cat, _) in CATEGORIAS.items():
    base[c] = (base[v] == cat).astype(float)

NUMERICOS = {'idade': 'Idade', 'reprovacao': 'Reprovação',
             'aluno_motivacao_aplicar_aprendizado': 'Q23b (motivação para usar o que foi ensinado)',
             'tempo_estudos': 'Tempo de estudos', 'tempo_cursos': 'Tempo em cursos',
             'tempo_trabalho_domestico': 'Trabalho doméstico', 'tempo_trabalho': 'Trabalho fora de casa',
             'mora_mae': 'Mora com a mãe', 'mora_pai': 'Mora com o pai',
             'escolaridade_mae': 'Escolaridade da mãe', 'escolaridade_pai': 'Escolaridade do pai',
             'responsaveis_incentivam_estudo': 'Responsáveis incentivam o estudo',
             'responsaveis_conversam_escola': 'Responsáveis conversam sobre a escola',
             'qtd_computadores': 'Computadores', 'possui_wifi': 'Wi-Fi',
             'possui_escrivaninha': 'Mesa para estudar', 'possui_casa_quarto_individual': 'Quarto só seu'}
UNIDADES = {**{c: r for c, (_, _, r) in CATEGORIAS.items()}, **NUMERICOS}
CONTROLES = ['em_integrado', 'nao_sabe_escolaridade_mae', 'nao_sabe_escolaridade_pai',
             'ausente_escolaridade_mae', 'ausente_escolaridade_pai',
             'ausente_aluno_motivacao_aplicar_aprendizado', 'ausente_mora_mae', 'ausente_mora_pai',
             'n_itens_ausentes']
COLS = list(UNIDADES) + CONTROLES
nu = len(UNIDADES)
print(nu, 'unidades comparadas')

25 unidades comparadas


## 2. Regressão

Redução do R² ao retirar cada unidade e valor-p do seu coeficiente (erros-padrão agrupados por escola). Retirar a coluna de uma categoria equivale a juntá-la à categoria de referência.

In [3]:
def formula(cols): return 'proficiencia ~ ' + ' + '.join(cols)
completo = smf.ols(formula(COLS), base).fit(cov_type='cluster', cov_kwds={'groups': base['id_escola']}, use_t=True)
reg = pd.DataFrame({'var': list(UNIDADES), 'Unidade': list(UNIDADES.values())})
reg['Redução do R²'] = [completo.rsquared - smf.ols(formula([c for c in COLS if c != v]), base).fit().rsquared for v in UNIDADES]
reg['p-valor'] = [completo.pvalues[v] for v in UNIDADES]
reg['Relevante (regressão)'] = reg['p-valor'] < 0.05
reg['Posição (regressão)'] = reg['Redução do R²'].rank(ascending=False).astype(int)
print(reg.sort_values('Posição (regressão)').drop(columns='var').to_string(index=False,
      formatters={'Redução do R²': '{:.5f}'.format, 'p-valor': '{:.2e}'.format}))

                                      Unidade Redução do R²  p-valor  Relevante (regressão)  Posição (regressão)
Q23b (motivação para usar o que foi ensinado)       0.02641 1.38e-75                   True                    1
                             Tempo de estudos       0.01072 9.48e-46                   True                    2
                 Pretensão: somente trabalhar       0.00718 1.74e-35                   True                    3
                              Raça/cor: preta       0.00583 1.47e-31                   True                    4
                              Raça/cor: parda       0.00578 2.77e-30                   True                    5
                               Sexo: feminino       0.00410 1.22e-21                   True                    6
                                   Reprovação       0.00390 1.39e-21                   True                    7
                           Raça/cor: indígena       0.00305 6.40e-17                   True     

## 3. RReliefF

Mesma implementação do Notebook 04 (todos os estudantes, k = 10), agora com as categorias como colunas binárias. A referência nula usa 20 permutações da proficiência (m = 5.000).

In [4]:
def rrelieff(X, y, k=10, m=None, seed=0):
    X = np.asarray(X, float); y = np.asarray(y, float); n, p = X.shape
    amp = X.max(0) - X.min(0); amp[amp == 0] = 1
    Xn = (X / amp).astype(np.float32); yn = y / (y.max() - y.min())
    rng = np.random.default_rng(seed)
    idx = np.arange(n) if (m is None or m >= n) else rng.choice(n, m, replace=False)
    NdC, NdA, NdCdA = 0.0, np.zeros(p), np.zeros(p)
    for i in idx:
        D = np.abs(Xn - Xn[i]); dist = D.sum(1); dist[i] = np.inf
        viz = np.argpartition(dist, k)[:k]
        dC = np.abs(yn[viz] - yn[i]); dA = D[viz].astype(float)
        NdC += dC.sum() / k; NdA += dA.sum(0) / k; NdCdA += (dC @ dA) / k
    M = len(idx)
    return NdCdA / NdC - (NdA - NdCdA) / (M - NdC)

inicio = time.time()
X_rr = base[COLS].values.astype(float)
w = rrelieff(X_rr, y, k=10)[:nu]
rng = np.random.default_rng(SEMENTE)
nulos = np.array([rrelieff(X_rr, rng.permutation(y), k=10, m=5000, seed=SEMENTE + 100 + b)[:nu] for b in range(20)])
rr = pd.DataFrame({'var': list(UNIDADES), 'Peso RReliefF': w, 'Limiar nulo (p95)': np.percentile(nulos, 95, axis=0)})
rr['Relevante (RReliefF)'] = rr['Peso RReliefF'] > rr['Limiar nulo (p95)']
rr['Posição (RReliefF)'] = rr['Peso RReliefF'].rank(ascending=False).astype(int)
print(f'Tempo: {time.time() - inicio:.0f} s')

Tempo: 174 s


## 4. Random Forest

Mesma partição por escola e mesma semente do Notebook 04. Cada unidade é embaralhada separadamente no conjunto de teste, 20 vezes.

In [5]:
inicio = time.time()
idx = np.arange(len(base))
tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEMENTE).split(idx, groups=base['id_escola']))
X = base[COLS].values.astype(float)
rf = RandomForestRegressor(**PARAMS_RF, random_state=SEMENTE).fit(X[tr], y[tr])
r2_base = r2_score(y[te], rf.predict(X[te]))
print(f'R² no teste: {r2_base:.4f}')
rng = np.random.default_rng(SEMENTE)
quedas = {}
for j, v in enumerate(UNIDADES):
    q = []
    for _ in range(20):
        Xp = X[te].copy(); Xp[:, j] = Xp[rng.permutation(len(te)), j]
        q.append(r2_base - r2_score(y[te], rf.predict(Xp)))
    quedas[v] = np.array(q)
rf_tab = pd.DataFrame({'var': list(UNIDADES), 'Importância média': [quedas[v].mean() for v in UNIDADES],
                       'Percentil 5': [np.percentile(quedas[v], 5) for v in UNIDADES]})
rf_tab['Relevante (Random Forest)'] = rf_tab['Percentil 5'] > 0
rf_tab['Posição (Random Forest)'] = rf_tab['Importância média'].rank(ascending=False).astype(int)
print(f'Tempo: {time.time() - inicio:.0f} s')

R² no teste: 0.1994
Tempo: 137 s


## 5. Comparação

In [6]:
comp = reg.merge(rr, on='var').merge(rf_tab, on='var')
pos = ['Posição (regressão)', 'Posição (RReliefF)', 'Posição (Random Forest)']
rel = ['Relevante (regressão)', 'Relevante (RReliefF)', 'Relevante (Random Forest)']
comp['Relevante nos 3'] = comp[rel].all(axis=1)
comp[f'Métodos no top {TOP}'] = (comp[pos] <= TOP).sum(axis=1)
comp['Convergente'] = comp['Relevante nos 3'] & (comp[f'Métodos no top {TOP}'] >= 2)
comp = comp.assign(media=comp[pos].mean(axis=1)).sort_values('media').drop(columns='media')
print(comp[['Unidade'] + pos + ['Relevante nos 3', f'Métodos no top {TOP}', 'Convergente']].to_string(index=False))
print('\nConvergentes (categorias separadas):', ', '.join(comp.loc[comp.Convergente, 'Unidade']) or 'nenhum')

def kendall_w(*r):
    R = np.column_stack(r); S = ((R.sum(1) - R.sum(1).mean()) ** 2).sum()
    return 12 * S / (R.shape[1] ** 2 * (R.shape[0] ** 3 - R.shape[0]))
a, b, c = (comp[p].values for p in pos)
conc = pd.DataFrame({'Medida': ['Regressão × RReliefF', 'Regressão × Random Forest', 'RReliefF × Random Forest',
                                'W de Kendall (três métodos)'],
                     'Categorias separadas (este notebook)': [spearmanr(a, b).statistic, spearmanr(a, c).statistic,
                                                              spearmanr(b, c).statistic, kendall_w(a, b, c)],
                     'Parecer (versão anterior, 27 unidades)': [0.441, 0.320, 0.585, np.nan]})
print(); print(conc.round(3).to_string(index=False))
print('\nOs valores do parecer foram calculados com os resultados da versão anterior (outra base e outro')
print('tratamento dos ausentes) e servem apenas de referência; o valor comparável do Notebook 04 está na aba "Concordância".')

with pd.ExcelWriter(PASTA + 'tabelas_comparacao_categorias.xlsx') as xw:
    comp.drop(columns='var').round(6).to_excel(xw, sheet_name='Comparação por categoria', index=False)
    conc.round(3).to_excel(xw, sheet_name='Concordância', index=False)
print('\nTabelas salvas em', PASTA + 'tabelas_comparacao_categorias.xlsx')

                                      Unidade  Posição (regressão)  Posição (RReliefF)  Posição (Random Forest)  Relevante nos 3  Métodos no top 5  Convergente
Q23b (motivação para usar o que foi ensinado)                    1                   1                        1             True                 3         True
                             Tempo de estudos                    2                   2                        2             True                 3         True
                 Pretensão: somente trabalhar                    3                  11                        3             True                 2         True
                                   Reprovação                    7                   5                        5             True                 2         True
                          Escolaridade da mãe                    9                   6                        6             True                 0        False
                                 Computa